# 05 — Submission

**One job:** select one frozen run, predict the test set, validate the CSV, and log it

Run from top to bottom. Slow actions are off by default.

In [ ]:
%pip install -q ultralytics pycocotools segmentation-models-pytorch

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json
import sys
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

PROJECT_DIR = Path("/content/drive/MyDrive/Google Colab/Solar Filament Segmentation Challenge")
sys.path.insert(0, str(PROJECT_DIR))
%load_ext autoreload
%autoreload 2

from src.config import DEFAULT_PIPELINE_SETTINGS, TRAINING_SETTINGS
from src.data import copy_raw_data_to_local, load_records, make_split, verify_or_write_split

LOCAL_DATA_DIR = copy_raw_data_to_local(PROJECT_DIR / "data/raw")
TRAIN_IMAGES_DIR = LOCAL_DATA_DIR / "train/train_images"
TEST_IMAGES_DIR = LOCAL_DATA_DIR / "test/test_images"
ANNOTATIONS_PATH = LOCAL_DATA_DIR / "train/MAGFiLO_1.0_Annotations_kaggle2026_train.json"
SPLIT_MANIFEST_PATH = PROJECT_DIR / "data/processed/split_manifest.json"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

In [ ]:
RUN_SUBMISSION = False
USE_ENSEMBLE = True  # seven-detector WBF + two averaged U-Nets (notebook 04 C4); False reproduces the single-model C0 submission
if USE_ENSEMBLE:
    from src.config import ENSEMBLE_SETTINGS
    RUN_ID = "20261027_212727_de_detector_ensemble_all7_t022_unet2"
    SUBMISSION_NAME = "wbf7_unet2"
    MASK_THRESHOLD = ENSEMBLE_SETTINGS["mask_threshold"]
else:
    RUN_ID = "20260909_230758_b5_c0"
    SUBMISSION_NAME = "b5_yolo26x"
    MASK_THRESHOLD = 0.45
YOLO_CHECKPOINT = PROJECT_DIR / "models/stage1/yolo26x_filament_detection/weights/best.pt"
UNET_CHECKPOINT = PROJECT_DIR / "models/stage2/crop_unet_b5/best_crop_unet_b5.pt"

## 1. Choose and inspect the run

In [ ]:
from src.experiments import load_registry

registry = load_registry(PROJECT_DIR / "experiments/registry.csv")
selected = registry.loc[registry["run_id"] == RUN_ID]
assert len(selected) == 1, f"Expected one registry row for {RUN_ID}"
display(selected)

## 2. Predict all 180 test images

In [ ]:
from src.inference import create_submission, validate_submission
from src.models import load_checkpoint

if RUN_SUBMISSION:
    from ultralytics import YOLO

    test_paths = sorted(TEST_IMAGES_DIR.glob("*.jpeg"))
    assert len(test_paths) == 180
    if USE_ENSEMBLE:
        from src.config import ENSEMBLE_UNETS
        from src.inference import ensemble_proposals
        from src.models import ModelAverage

        proposals = ensemble_proposals(PROJECT_DIR / "models", test_paths, device, DEFAULT_PIPELINE_SETTINGS)
        unet = ModelAverage([load_checkpoint(PROJECT_DIR / path, "b5", device)[0] for path in ENSEMBLE_UNETS]).to(device).eval()
        submission, image_statistics = create_submission(test_paths, None, unet, device, MASK_THRESHOLD, DEFAULT_PIPELINE_SETTINGS.copy(), proposals)
    else:
        detector = YOLO(str(YOLO_CHECKPOINT))
        unet, checkpoint = load_checkpoint(UNET_CHECKPOINT, "b5", device)
        submission, image_statistics = create_submission(test_paths, detector, unet, device, MASK_THRESHOLD, DEFAULT_PIPELINE_SETTINGS.copy())
else:
    print("Submission inference skipped.")

## 3. Validate and list zero-mask images

Validation checks unique ids, exact 2048×2048 decoding, and no overlap within an image.

In [ ]:
if RUN_SUBMISSION:
    assert validate_submission(submission)
    zero_mask_images = image_statistics.query("final_instances == 0")["image"].tolist()
    print(f"Rows: {len(submission)}, zero-mask images: {zero_mask_images}")

## 4. Save without overwriting and append the log

In [ ]:
if RUN_SUBMISSION:
    output_dir = PROJECT_DIR / "submissions" / datetime.now().strftime(f"%Y%m%d_%H%M%S_{SUBMISSION_NAME}")
    output_dir.mkdir(parents=True, exist_ok=False)
    submission.to_csv(output_dir / "submission.csv", index=False)
    image_statistics.to_csv(output_dir / "image_statistics.csv", index=False)
    (output_dir / "settings.json").write_text(json.dumps({"run_id": RUN_ID, "mask_threshold": MASK_THRESHOLD, "pipeline": DEFAULT_PIPELINE_SETTINGS,
                                                           "ensemble": ENSEMBLE_SETTINGS if USE_ENSEMBLE else None}, indent=2))
    log_path = PROJECT_DIR / "submissions/leaderboard_log.csv"
    row = pd.DataFrame([{"date": datetime.now().date(), "run_id": RUN_ID, "local_pq": float(selected.iloc[0]["official_pq"]),
                         "kaggle_public": "", "submission_path": str(output_dir.relative_to(PROJECT_DIR)), "notes": "Kaggle score pending"}])
    row.to_csv(log_path, mode="a", header=not log_path.exists(), index=False)
    print(output_dir)